# 0. 先导：为什么基础 RAG 还不够

本章目标：用可运行的失败案例，说明基础 RAG 在真实场景中的三类典型不足，并为后续三层增强（上下文 / 流程 / 系统）建立统一问题框架。

你将看到：同一套数据与问题下，基础 RAG 往往“相关但不完整、一次不够、跨轮易丢信息”。

## 基础 RAG 的默认假设（也是后续失败根源）

基础 RAG 往往隐含以下假设：

1. **一次检索就够**：用户问题可以被单轮召回覆盖。
2. **命中片段就可回答**：被召回块内部已包含充分上下文。
3. **当前问题独立存在**：不依赖多轮历史、跨文档关系和系统状态。

当这三个假设不成立时，就需要进入第六章的增强方法。

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# LangChain fallback: 当前环境未安装 llama_index，先保证最小可运行示例。
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

load_dotenv()

BASE_DIR = Path("notebook/C7 高级 RAG 技巧/6. 增强阶段")
FACE_PDF = BASE_DIR / "data" / "face.pdf"
MODEL_NAME = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def build_retriever(chunk_size: int = 512, chunk_overlap: int = 80, k: int = 4):
    docs = PyPDFLoader(str(FACE_PDF)).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
    return vectorstore.as_retriever(search_kwargs={"k": k}), chunks

def show_retrieved_docs(docs):
    for i, doc in enumerate(docs, 1):
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"[{i}] {preview}...")

## 失败案例 1：检索“相关”但上下文仍不完整（上下文增强动机）

我们故意把 chunk 切得很小（约 128），再问一个需要跨句整合的问题。

预期现象：
- 能召回“相关句子”；
- 但证据分散在多个块里，最终答案容易缺要点。

In [ ]:
q1 = "请总结 face 文档里关于关键问题成因与改进方向的完整链路。"
retriever_small, _ = build_retriever(chunk_size=128, chunk_overlap=16, k=4)
retrieved_q1 = retriever_small.invoke(q1)

print("=== 检索片段（小块）===")
show_retrieved_docs(retrieved_q1)

context_q1 = "\n\n".join(doc.page_content for doc in retrieved_q1)
prompt_q1 = f"""
你是严谨的问答助手。仅基于给定上下文回答。

问题：{q1}

上下文：
{context_q1}

请给出结构化回答：
1) 关键成因
2) 证据
3) 改进方向
"""

ans_q1 = llm.invoke(prompt_q1).content
print("\n=== 基础 RAG 回答 ===")
print(ans_q1)

### 失败案例 1 分析

这个失败本质是“**证据分布在相邻或跨段上下文中**”，而不是“检索完全没命中”。

因此下一步应该做的是**上下文增强**（如 Sentence Window、Small-to-Big、AutoMerging），而不是直接引入复杂流程或系统编排。

→ 解法详见 **1. 上下文增强（重构版）**


## 失败案例 2：一次检索 + 一次生成不够（流程增强动机）

问题如果天然需要“拆解 -> 补检索 -> 再综合”，单轮流程通常只能给出部分答案。

预期现象：
- 回答方向正确；
- 但缺少子问题维度，覆盖不完整。

In [ ]:
q2 = "请先识别 face 文档中的核心问题，再分别说明每个问题对应的改进建议与落地顺序。"
retriever_base, _ = build_retriever(chunk_size=512, chunk_overlap=80, k=3)
retrieved_q2 = retriever_base.invoke(q2)

print("=== 检索片段（单轮）===")
show_retrieved_docs(retrieved_q2)

context_q2 = "\n\n".join(doc.page_content for doc in retrieved_q2)
prompt_q2 = f"""
仅基于上下文回答。
问题：{q2}
上下文：{context_q2}
"""

ans_q2 = llm.invoke(prompt_q2).content
print("\n=== 单轮流程回答 ===")
print(ans_q2)

### 失败案例 2 分析

这个失败不是“块不够大”，而是“**流程少了一步**”：
- 需要先做子问题拆解或迭代检索；
- 再做跨子问题证据合成。

因此它属于**流程增强**问题（迭代检索、递归检索、查询路由、Corrective RAG、Self-RAG）。

→ 解法详见 **2. 流程增强（重构版）**


## 失败案例 3：多文档 / 多轮问题下状态丢失（系统增强动机）

这里演示一个简化版多轮失败：第二问使用代词“它”，若系统没有会话记忆，容易答错对象。

预期现象：
- 第一问可答；
- 第二问在无记忆条件下出现语义漂移。

In [ ]:
city_docs = [
    "北京：科技与教育资源密集，创新产业聚集。",
    "上海：国际金融与航运中心，开放型经济特征明显。",
]

turn1_q = "请比较北京和上海在产业定位上的差异。"
turn2_q = "那它在国际化方面的优势是什么？"

# 无记忆：第二问单独处理，代词“它”无指代解析。
prompt_turn2_wo_memory = f"""
只看当前问题，不参考历史：{turn2_q}
候选资料：{city_docs}
请作答。
"""

ans_turn2_wo_memory = llm.invoke(prompt_turn2_wo_memory).content
print("=== 第二问（无记忆）===")
print(ans_turn2_wo_memory)

# 有最小记忆：把第一轮问题拼入上下文。
prompt_turn2_with_memory = f"""
历史问题：{turn1_q}
当前问题：{turn2_q}
候选资料：{city_docs}
请先判断“它”指代哪个城市，再回答国际化优势。
"""

ans_turn2_with_memory = llm.invoke(prompt_turn2_with_memory).content
print("\n=== 第二问（带最小记忆）===")
print(ans_turn2_with_memory)

### 失败案例 3 分析

这类问题核心不是“检索不到”，而是“**系统缺状态管理**”：
- 需要会话记忆（短期/长期）；
- 需要跨文档工具编排与路由；
- 需要可观测的执行链路。

因此它属于**系统增强**问题。

→ 解法详见 **3. 系统增强**


## 本章导航

| 失败类型 | 代表案例 | 解法所在 | 核心方法 |
|---|---|---|---|
| 检索相关但上下文不全 | 案例 1 | 1. 上下文增强 | Sentence Window / Small-to-Big / AutoMerging |
| 一次检索+一次生成不够 | 案例 2 | 2. 流程增强 | 迭代检索 / 递归检索 / CRAG / Self-RAG / 自适应检索 |
| 多轮/多文档/状态丢失 | 案例 3 | 3. 系统增强 | Memory / Multi-Doc Agent / Agentic RAG / GraphRAG |


## 方法地图

```mermaid
flowchart TD
    basic[基础 RAG] --> ctx[上下文增强]
    basic --> flow[流程增强]
    basic --> sys[系统增强]

    ctx --> sw[Sentence Window]
    ctx --> stb[Small-to-Big]
    ctx --> am[AutoMerging]

    flow --> ir[迭代检索]
    flow --> rr[递归检索]
    flow --> qr[查询路由与自适应检索]
    flow --> crag[Corrective RAG]
    flow --> selfrag[Self-RAG]

    sys --> mem[Memory]
    sys --> mda[Multi-Document Agent]
    sys --> agentic[Agentic RAG]
    sys --> graphrag[GraphRAG]
```

## 学习路径

1. 先学 `1. 上下文增强（重构版）`：解决“检索相关但上下文不全”。
2. 再学 `2. 流程增强（重构版）`：解决“一轮流程不够”的动态检索与决策。
3. 最后学 `3. 系统增强`：解决多轮、多文档、可观测、可恢复等工程问题。
4. 用 `4. 选型总结.md` 做方法组合与成本权衡。

## 本章边界

- 不重复第 3~5 章的基础构建与优化细节。
- 本章重点是：当基础 RAG 已上线后，如何系统性定位失败类型并选择增强层级。
- 所有示例优先教学可读性与最小可运行，不追求生产级性能调优。